In [1]:
import os
os.chdir('../../../..')

In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

from src.datasets import QM9Dataset

In [3]:
qm9 = QM9Dataset(limit=5_000, sampling_strategy="stratified", stratify_by=["num_atoms", "gap"], add_acsf=True, add_soap=True, add_chemprop=True, add_selfies_transformer=True, add_selfies_onehot=True, add_morgan_fingerprint=True)
df = qm9.load()

2026-04-17 16:09:04.367 | INFO     | src.datasets:load:494 - Loading cached full QM9 dataset from: data/QM9/dataset_cleaned.parquet
2026-04-17 16:09:04.966 | INFO     | src.datasets:_sample_qm9_df:686 - QM9 sampling complete: strategy=stratified, requested_limit=5500, returned_rows=5500.
2026-04-17 16:09:04.968 | INFO     | src.datasets:_add_requested_descriptors:127 - Applying requested QM9 descriptors to sampled dataframe (rows=5500).
2026-04-17 16:09:04.968 | INFO     | src.features:compute_morgan_fingerprints:76 - Computing Morgan Fingerprints (Radius=3, Size=2048)...
2026-04-17 16:09:14.019 | INFO     | src.features:compute_selfies_transformer:93 - Computing SELFormer Embeddings using HUBioDataLab/SELFormer...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

2026-04-17 16:10:10.836 | INFO     | src.features:compute_selfies_transformer:132 - Reducing dimensions from 768 to 32 using PCA...
2026-04-17 16:10:11.276 | INFO     | src.features:compute_selfies_onehot:142 - Computing One-Hot Encodings...
2026-04-17 16:10:12.474 | INFO     | src.features:compute_soap:170 - Computing SOAP (rcut=6.0, nmax=8, lmax=6)...
2026-04-17 16:11:10.358 | SUCCESS  | src.datasets:add_soap:840 - Added SOAP embeddings.
2026-04-17 16:11:10.359 | INFO     | src.features:compute_acsf:201 - Computing ACSF (rcut=6.0)...
2026-04-17 16:11:50.869 | SUCCESS  | src.datasets:add_acsf:851 - Added ACSF embeddings.
2026-04-17 16:11:50.870 | INFO     | src.features:compute_chemprop_embeddings:284 - Computing Chemprop embeddings on mps...
2026-04-17 16:11:50.871 | WARNING  | src.features:compute_chemprop_embeddings:292 - No model_path provided. Using RANDOM (untrained) MPNN weights.
2026-04-17 16:11:54.549 | INFO     | src.datasets:_add_requested_descriptors:150 - Added descriptor

In [4]:
onehot_matrix = qm9.get_distance_matrix(
    descriptor='onehot',
    dist_type="jaccard",
    force_calculate=True
)

morgan_matrix = qm9.get_distance_matrix(
    descriptor='morgan',
    dist_type="jaccard",
    force_calculate=True
)

soap_matrix = qm9.get_distance_matrix(
    descriptor='soap',
    dist_type="soap_kernel",
    force_calculate=True
)

acsf_matrix = qm9.get_distance_matrix(
    descriptor='acsf',
    dist_type="euclidean",
    force_calculate=True
)

transformer_matrix = qm9.get_distance_matrix(
    descriptor='transformer',
    dist_type="euclidean",
    force_calculate=True
)

chemprop_matrix = qm9.get_distance_matrix(
    descriptor='chemprop',
    dist_type="euclidean",
    force_calculate=True
)

2026-04-17 16:11:54.723 | INFO     | src.datasets:get_distance_matrix:1002 - Calculating distance matrix for selfies_onehot using jaccard distance.
2026-04-17 16:12:20.288 | SUCCESS  | src.distance:_compute_and_save:79 - Saved distance matrix to data/QM9/dist_selfies_onehot_jaccard.npy
2026-04-17 16:12:20.302 | INFO     | src.datasets:get_distance_matrix:1002 - Calculating distance matrix for morgan using jaccard distance.
2026-04-17 16:12:38.291 | SUCCESS  | src.distance:_compute_and_save:79 - Saved distance matrix to data/QM9/dist_morgan_jaccard.npy
2026-04-17 16:12:38.299 | INFO     | src.datasets:get_distance_matrix:1002 - Calculating distance matrix for soap using soap_kernel distance.
2026-04-17 16:12:38.975 | SUCCESS  | src.distance:_compute_and_save:79 - Saved distance matrix to data/QM9/dist_soap_soap_kernel.npy
2026-04-17 16:12:39.036 | INFO     | src.datasets:get_distance_matrix:1002 - Calculating distance matrix for acsf using euclidean distance.
2026-04-17 16:12:39.318 | S

In [ ]:
import numpy as np
import pandas as pd
from itertools import combinations
from joblib import Parallel, delayed
from tqdm import tqdm
from skbio.stats.distance import mantel

matrices = {
    "OneHot": onehot_matrix,
    "Morgan": morgan_matrix,
    "SOAP": soap_matrix,
    "ACSF": acsf_matrix,
    "Transformer": transformer_matrix,
    "Chemprop": chemprop_matrix
}

names = list(matrices.keys())
n = len(names)

# Initialize results matrix with 1.0 on the diagonal 
# (a metric perfectly correlates with itself)
results = np.zeros((n, n))
np.fill_diagonal(results, 1.0)

# Generate all unique pairs of indices (e.g., (0,1), (0,2)...)
# This prevents calculating A vs B and then redundantly calculating B vs A
pairs = list(combinations(range(n), 2))

# Define the worker function for the parallel pool
def compute_pair(i, j):
    """
    Runs the scikit-bio mantel test on a pair of matrices.
    """
    # 1. Cast the matrices to 64-bit floats (C 'double'). 
    # This simultaneously creates a writable copy, bypassing joblib's lock.
    mat_i = matrices[names[i]].astype(np.float64)
    mat_j = matrices[names[j]].astype(np.float64)
    
    # 2. Ensure the matrices are strictly "hollow" (diagonal == 0.0)
    np.fill_diagonal(mat_i, 0.0)
    np.fill_diagonal(mat_j, 0.0)

    # 3. Run the optimized mantel test
    r, p, _ = mantel(
        mat_i, 
        mat_j, 
        method='pearson', 
        permutations=999 
    )
    return i, j, r

print(f"Executing {len(pairs)} pairwise Mantel comparisons...")

# Execute in parallel across all available CPU cores (n_jobs=-1)
results = [compute_pair(i,j) for i,j in tqdm(pairs, desc="Computing pairs")]

print("Unpacking the results")
# Unpack the results and mirror them across the diagonal
for i, j, r in results:
    results[i, j] = results[j, i] = r

# Convert to DataFrame for visualization (e.g., seaborn.heatmap)
mantel_df = pd.DataFrame(results, index=names, columns=names)

print("\nFinal Mantel Correlation Matrix:")
print(mantel_df)

Executing 15 pairwise Mantel comparisons...


Computing pairs:   0%|          | 0/15 [00:00<?, ?it/s]